# 15 - accDM daughter: fluid-closure diagnostic + two-regime ceff2 fit

Two parts:
1. **Diagnostic (Tasks 1-5):** does the effective sound speed / shear closure collapse onto one function of `x=k/k_fs`? Pole-masked raw `delta_p/delta_rho` and `sigma/delta`. Finding: universal at `x<1`, splits by **eta (not f)** at `x>1`, so `f<=0.3` is not the obstacle.
2. **Fit (Task 6):** the measured `ceff2` is two-regime - adiabatic `ca2` below a transition `x_t`, a free-streaming plateau `c_fs(eta)` above - and `c_fs` **increases** with `eta` (the published weight `W=1-2*eps_acc` decreases with eta, i.e. has the wrong sign). Fit `ceff2(x;eta)=ca2*(1-S)+c_fs(eta)*S`, `S=x^p/(x_t^p+x^p)`.

Spec/plan under `docs/superpowers/`. Style follows notebook 7. `w_sigma`/`w_theta` are zero in the exact hierarchy (`memory: w-sigma-zero-in-exact-hierarchy`), so the shear sector uses `sigma/delta` and `k*sigma/theta`.

**Decisive follow-up (Phase 3, after a rebuild):** validate the fit on **P(k) over k<=1 only** - notebook 7 failed because it included k<=10 where the fluid blows up architecturally.

In [ ]:
import sys; sys.path.insert(0, '.')          # import helpers from notebooks_test/
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from classy import Class
from fluid_closure_helpers import (
    ca2_from_kfs, mask_small_denom, log_upper_envelope, collapse_band,
    smooth_step, two_regime_ceff2,
)

plt.rcParams.update({
    'mathtext.fontset': 'stix', 'font.family': 'serif', 'font.size': 11,
    'axes.labelsize': 12, 'legend.fontsize': 9, 'lines.linewidth': 1.5, 'figure.dpi': 200})
qual_colors = ['#377eb8', '#ff7f00', '#4daf4a', '#f781bf', '#984ea3']

# --- base cosmology + accDM model (same setup as notebook 7) ---
omega_b, omega_cdm0 = 0.022383, 0.12011
A_s, n_s, tau_reio, H0 = 2.1005829616811546e-9, 0.96605, 0.0543, 67.32
base_params = {'omega_b': omega_b, 'omega_cdm': omega_cdm0, 'H0': H0,
               'A_s': A_s, 'n_s': n_s, 'tau_reio': tau_reio}
PREC = {'output': 'mPk', 'P_k_max_1/Mpc': 10.0, 'z_max_pk': 0.0,
        'evolver': 0, 'reionization_z_start_max': 80}

A_T, MASS, KAPPA = 0.13, 1e16, 6.0
A_REC = 1.0 / (1.0 + 1090.0)

# --- diagnostic sweep grids ---
K_GRID   = np.logspace(-2, 0.0, 18)          # 1/Mpc: sub-horizon, spans below to above k_fs
F_LIST   = [0.05, 0.1, 0.2, 0.3]             # daughter fraction up to the target
ETA_LIST = [0.1, 0.3, 1.0]                    # 3 boosts -> pin c_fs(eta)
DROP_FRAC = 0.2                               # pole mask: drop smallest 20% of |denominator|

def eps_acc_of_eta(eta):
    e2 = eta*eta
    return -e2 + np.sqrt(e2*e2 + 4*e2*eta + 5*e2 + 2*eta) - 2*eta

def accdm_params(eta, f_acc=0.1, kappa=KAPPA):
    """Exact-hierarchy accDM params for a given boost eta and daughter fraction f_acc."""
    ocdm = omega_cdm0 * (1 + f_acc*(1 - A_REC**kappa)/(1 + (A_REC/A_T)**kappa))**(-1)
    p = dict(base_params); p.update(PREC)
    p.update({'omega_cdm': ocdm,
              'vary_Gamma_acc': 'yes', 'kappa_acc': kappa, 'a_t_acc': A_T,
              'f_acc': f_acc, 'eta_acc': eta,
              'm_acc_in_GeV': MASS, 'm_cdm_in_GeV': MASS,
              'N_ncdm': 2, 'deg_ncdm': '3, 1',
              'm_ncdm': '0.02, {:.6e}'.format(MASS*1e9),
              'T_ncdm': '0.71611, 1', 'ncdm_quadrature_strategy': '0, 4',
              'ncdm_N_momentum_bins': '15, 501', 'N_ur': 0.00441,
              'background_Nloga': 5001, 'gauge': 'synchronous',
              'get_perturbations_in_current_gauge': 'yes',
              'ncdm_fluid_trigger_tau_over_tau_k': 25,
              'ncdm_fluid_approximation': 3})          # 3 = none (exact hierarchy)
    return p

print('setup OK; W(eta):', {e: round(1 - 2*eps_acc_of_eta(e), 3) for e in ETA_LIST},
      '<- decreases with eta (data increases -> W has wrong sign)')

## Task 2 - tau-series extractor

Exact hierarchy, full tau-series of `{delta, theta, shear, delta_p/delta_rho, k_fs}` per k; `aH(tau)` from the background, `ca2 = (3/2)(aH/k_fs)^2`.

In [ ]:
def _find_key(d, want):
    if want in d:
        return want
    for kk in d:
        if kk.replace(' ', '').startswith(want.replace(' ', '')):
            return kk
    raise KeyError('{!r} not found; available: {}'.format(want, list(d.keys())))

def extract_daughter_series(params, k_list):
    """Run exact CLASS; return {k: dict of daughter tau-series arrays}."""
    ks = np.sort(np.asarray(k_list, float))
    p = dict(params); p['k_output_values'] = ', '.join('{:.8e}'.format(k) for k in ks)
    M = Class(); M.set(p); M.compute()
    perts = M.get_perturbations()['scalar']
    bg = M.get_background()
    tau_bg = np.asarray(bg['conf. time [Mpc]'], float)
    a_bg   = 1.0 / (1.0 + np.asarray(bg['z'], float))
    H_bg   = np.asarray(bg['H [1/Mpc]'], float)
    o = np.argsort(tau_bg); tau_bg, a_bg, H_bg = tau_bg[o], a_bg[o], H_bg[o]
    kd  = _find_key(perts[0], 'delta_ncdm[1]'); kt  = _find_key(perts[0], 'theta_ncdm[1]')
    ksh = _find_key(perts[0], 'shear_ncdm[1]'); kc  = _find_key(perts[0], 'cs2_ncdm[1]')
    kf  = _find_key(perts[0], 'k_fss_acc[1]');  ktau = _find_key(perts[0], 'tau')
    out = {}
    for k, d in zip(ks, perts):
        tau = np.asarray(d[ktau], float)
        aH  = np.interp(tau, tau_bg, a_bg) * np.interp(tau, tau_bg, H_bg)
        out[k] = dict(tau=tau, aH=aH,
                      delta=np.asarray(d[kd], float), theta=np.asarray(d[kt], float),
                      shear=np.asarray(d[ksh], float), dpr=np.asarray(d[kc], float),
                      k_fs=np.asarray(d[kf], float))
    M.struct_cleanup(); M.empty()
    return out

In [ ]:
_probe = extract_daughter_series(accdm_params(ETA_LIST[0], f_acc=0.1), K_GRID[:3])
_s = _probe[K_GRID[0]]
for q in ('theta', 'shear', 'dpr', 'k_fs', 'aH', 'tau'):
    assert _s[q].shape == _s['delta'].shape, q
g = (_s['k_fs'] > 0) & np.isfinite(_s['aH'])
ca2 = ca2_from_kfs(_s['k_fs'][g], _s['aH'][g])
print('extractor OK; tau samples per k =', _s['delta'].size,
      '| ca2 in [{:.2e},{:.2e}] | x=k/k_fs reaches {:.0f}'.format(
          ca2.min(), ca2.max(), (K_GRID.max()/_s['k_fs'][g]).max()))
assert np.all(ca2 <= 1.0 + 1e-6), 'ca2 > 1 -> aH/k_fs mismatch (check background keys)'

## Task 3 - run the sweep once, cache series, build response envelopes

One exact run per `(f, eta)`; the full extraction is cached in `SERIES` so Task 6's fit reuses it (no second CLASS pass). `RESPONSES` holds the pole-masked envelopes of `ceff2`, `sigma/delta`, and `k*sigma/theta`.

In [ ]:
def responses_from_series(ser, eta, drop_frac=DROP_FRAC):
    W = 1 - 2*eps_acc_of_eta(eta)
    Xc, Ce, Cf, Sd, Xv, Rv = [], [], [], [], [], []
    for k, s in ser.items():
        g = np.isfinite(s['k_fs']) & (s['k_fs'] > 0) & np.isfinite(s['aH'])
        if not np.any(g):
            continue
        x   = k / s['k_fs'][g]
        ca2 = ca2_from_kfs(s['k_fs'][g], s['aH'][g])
        delta = s['delta'][g]; theta = s['theta'][g]
        shear = s['shear'][g]; dpr = s['dpr'][g]
        safe_d = np.where(delta == 0, np.nan, delta)
        safe_t = np.where(theta == 0, np.nan, theta)
        Xc.append(x); Ce.append(np.abs(mask_small_denom(dpr, delta, drop_frac)))
        Cf.append(ca2 * (1 + 0.2*W*np.sqrt(x)))
        Sd.append(np.abs(mask_small_denom(shear/safe_d, delta, drop_frac)))
        Xv.append(x); Rv.append(np.abs(mask_small_denom(k*shear/safe_t, theta, drop_frac)))
    cat = np.concatenate
    return {'ceff2':     log_upper_envelope(cat(Xc), cat(Ce)),
            'ceff2_fit': log_upper_envelope(cat(Xc), cat(Cf)),
            'sig_del':   log_upper_envelope(cat(Xc), cat(Sd)),
            'Rv':        log_upper_envelope(cat(Xv), cat(Rv))}

SERIES, RESPONSES = {}, {}
for eta in ETA_LIST:
    for f in tqdm(F_LIST, desc='eta={}'.format(eta)):
        ser = extract_daughter_series(accdm_params(eta, f_acc=f), K_GRID)
        SERIES[(f, eta)] = ser
        RESPONSES[(f, eta)] = responses_from_series(ser, eta)
print('cached', len(SERIES), 'series / responses')

## Task 4 - collapse plots + A/B/C decision

In [ ]:
X_EVAL = np.logspace(-1, 3, 60)
ls_eta = {ETA_LIST[0]: '-', ETA_LIST[1]: '-.', ETA_LIST[2]: '--'}
col_f  = {f: qual_colors[i] for i, f in enumerate(F_LIST)}

def band_all(w):
    return collapse_band(X_EVAL, [RESPONSES[k][w] for k in RESPONSES])[1]
def band_fixed_eta(w):   # pool over f at fixed eta (the relevant grouping: dependence is on eta)
    return float(np.nanmax([collapse_band(X_EVAL, [RESPONSES[(f, e)][w] for f in F_LIST])[1]
                            for e in ETA_LIST]))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)
for (f, eta), r in RESPONSES.items():
    for ax, w in zip(axes, ('ceff2', 'sig_del')):
        x, env = r[w]
        if x.size:
            ax.loglog(x, env, ls_eta[eta], color=col_f[f], alpha=0.8,
                      label='f={}, eta={}'.format(f, eta))
xf, ef = RESPONSES[(0.1, 0.1)]['ceff2_fit']
axes[0].loglog(xf, ef, 'k:', label=r'paper-form $c_a^2(1{+}0.2W\sqrt{x})$')
axes[0].axhline(1./3., color='red', ls='--', lw=1.0, label=r'causal $1/3$')
axes[0].set_title(r'raw $c_{\rm eff}^2=\delta p/\delta\rho$ (pole-masked)')
axes[1].set_title(r'$\sigma/\delta$ (shear closure, pole-masked)')
for ax in axes:
    ax.set_xlabel(r'$x=k/k_{\rm fs}$'); ax.grid(True, which='both', alpha=0.3)
axes[0].legend(fontsize=6, ncol=2); plt.show()

band = {w: (band_all(w), band_fixed_eta(w)) for w in ('ceff2', 'sig_del')}
print('band (pooled-all, fixed-eta):')
for w in ('ceff2', 'sig_del'):
    print('  {:>8}: {:.3f}, {:.3f}'.format(w, *band[w]))

In [ ]:
TOL = 0.15
def verdict(w):
    pooled, fixede = band[w]
    if not np.isfinite(pooled):
        return 'NO DATA'
    if pooled < TOL:
        return 'UNIVERSAL in x'
    if np.isfinite(fixede) and fixede < TOL:
        return 'ETA-PARAMETRIZED (collapses over f at fixed eta)'
    return 'ENVELOPE-ONLY (oscillation-limited; needs P(k) test)'
print('ceff2 :', verdict('ceff2'))
print('sig/del:', verdict('sig_del'))
print('\n-> f<=0.3 is not the obstacle; eta is the knob. Proceed to the two-regime fit (Task 6).')

## Task 5 - where does collapse hold? (band vs x) + kinematic cross-check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)
bc, _ = collapse_band(X_EVAL, [RESPONSES[k]['ceff2']   for k in RESPONSES])
bs, _ = collapse_band(X_EVAL, [RESPONSES[k]['sig_del'] for k in RESPONSES])
axes[0].loglog(X_EVAL, bc, '-', color=qual_colors[0], label=r'$c_{\rm eff}^2$ band$(x)$')
axes[0].loglog(X_EVAL, bs, '-', color=qual_colors[2], label=r'$\sigma/\delta$ band$(x)$')
axes[0].axhline(TOL, color='red', ls='--', lw=1.0, label='TOL=0.15')
axes[0].axvline(1.0, color='gray', ls=':', lw=1.0, label=r'$x=1$')
axes[0].set_title('collapse band vs x (pooled over f, eta)')
axes[0].set_xlabel(r'$x=k/k_{\rm fs}$'); axes[0].set_ylabel('fractional band width')
axes[0].grid(True, which='both', alpha=0.3); axes[0].legend(fontsize=8)
for (f, eta), r in RESPONSES.items():
    x, env = r['Rv']
    if x.size:
        axes[1].loglog(x, env, ls_eta[eta], color=col_f[f], alpha=0.8)
axes[1].set_title(r'cross-check $R_v=k\sigma/\theta$ (pole-masked)')
axes[1].set_xlabel(r'$x=k/k_{\rm fs}$'); axes[1].grid(True, which='both', alpha=0.3)
plt.show()

## Task 6 - fit the two-regime `ceff2(x; eta)`

Pool the pole-masked `(x, ca2, ceff2)` samples over `f` at each `eta` (from the `SERIES` cache), take robust medians per log-x bin, and fit `ceff2 = ca2*(1-S) + c_fs*S`, `S=x^p/(x_t^p+x^p)`, with **shared `(x_t, p)`** and **per-eta `c_fs`**. Grid-search `(x_t, p)` with a closed-form (relative-weighted) `c_fs` inner solve - pure numpy, no scipy.

In [ ]:
def binned_samples(eta, n_bins=30, min_per_bin=4):
    """Pool pole-masked (x, ca2, ceff2) over f at this eta; robust median per log-x bin."""
    xs, cas, ces = [], [], []
    for f in F_LIST:
        for k, s in SERIES[(f, eta)].items():
            g = np.isfinite(s['k_fs']) & (s['k_fs'] > 0) & np.isfinite(s['aH'])
            if not np.any(g):
                continue
            x = k / s['k_fs'][g]
            ca2 = ca2_from_kfs(s['k_fs'][g], s['aH'][g])
            ce = np.abs(mask_small_denom(s['dpr'][g], s['delta'][g], DROP_FRAC))
            m = np.isfinite(ce) & (ce > 0)
            xs.append(x[m]); cas.append(ca2[m]); ces.append(ce[m])
    x = np.concatenate(xs); ca2 = np.concatenate(cas); ce = np.concatenate(ces)
    edges = np.logspace(np.log10(x.min()), np.log10(x.max()), n_bins + 1)
    idx = np.clip(np.digitize(x, edges) - 1, 0, n_bins - 1)
    xb, cab, deb = [], [], []
    for b in range(n_bins):
        sel = idx == b
        if sel.sum() >= min_per_bin:
            xb.append(np.sqrt(edges[b]*edges[b+1]))
            cab.append(np.median(ca2[sel])); deb.append(np.median(ce[sel]))
    return np.array(xb), np.array(cab), np.array(deb)

BIN = {eta: binned_samples(eta) for eta in ETA_LIST}

def _fit_cfs(xb, cab, deb, x_t, p):
    """Closed-form relative-weighted c_fs for given (x_t,p); returns (c_fs, weighted resid)."""
    S = smooth_step(xb, x_t, p); w = 1.0/deb**2
    den = np.sum(w*S*S)
    c_fs = np.sum(w*S*(deb - cab*(1-S))) / den if den > 0 else 0.0
    model = cab*(1-S) + c_fs*S
    return c_fs, float(np.sum(w*(model - deb)**2))

X_T_GRID = np.logspace(0, np.log10(30), 25)
P_GRID   = np.linspace(0.5, 4.0, 20)
best = None
for x_t in X_T_GRID:
    for p in P_GRID:
        tot, cfs = 0.0, {}
        for eta in ETA_LIST:
            c, r = _fit_cfs(*BIN[eta], x_t, p); cfs[eta] = c; tot += r
        if best is None or tot < best[0]:
            best = (tot, x_t, p, cfs)
_, X_T, P, CFS = best
print('fit: x_t = {:.2f}, p = {:.2f}'.format(X_T, P))
for eta in ETA_LIST:
    print('  c_fs(eta={}) = {:.3f}'.format(eta, CFS[eta]))
print('c_fs monotone increasing in eta:', bool(np.all(np.diff([CFS[e] for e in ETA_LIST]) > 0)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)
# (left) two-regime fit vs binned data + old sqrt/W form
for eta, c in zip(ETA_LIST, qual_colors):
    xb, cab, deb = BIN[eta]
    axes[0].loglog(xb, deb, 'o', color=c, ms=4, label='data eta={}'.format(eta))
    axes[0].loglog(xb, two_regime_ceff2(xb, cab, CFS[eta], X_T, P), '-', color=c)
    W = 1 - 2*eps_acc_of_eta(eta)
    axes[0].loglog(xb, np.abs(cab*(1 + 0.2*W*np.sqrt(xb))), ':', color=c, alpha=0.6)
axes[0].axhline(1./3., color='red', ls='--', lw=1.0)
axes[0].axvline(X_T, color='gray', ls=':', lw=1.0, label='x_t={:.1f}'.format(X_T))
axes[0].set_title('two-regime fit (-) vs data (o) vs old sqrt/W (:)')
axes[0].set_xlabel(r'$x=k/k_{\rm fs}$'); axes[0].set_ylabel(r'$c_{\rm eff}^2$')
axes[0].grid(True, which='both', alpha=0.3); axes[0].legend(fontsize=7)
# (right) c_fs(eta)
etas = np.array(ETA_LIST); cvals = np.array([CFS[e] for e in ETA_LIST])
axes[1].plot(etas, cvals, 'o-', color=qual_colors[0])
axes[1].set_title(r'$c_{\rm fs}(\eta)$ (increases with $\eta$)')
axes[1].set_xlabel(r'$\eta$'); axes[1].set_ylabel(r'$c_{\rm fs}$'); axes[1].grid(alpha=0.3)
plt.show()

# quantify improvement: log-RMS residual of new vs old form on the binned medians
print('log-RMS residual (lower is better):')
for eta in ETA_LIST:
    xb, cab, deb = BIN[eta]; W = 1 - 2*eps_acc_of_eta(eta)
    r_new = np.sqrt(np.mean((np.log(two_regime_ceff2(xb, cab, CFS[eta], X_T, P)) - np.log(deb))**2))
    r_old = np.sqrt(np.mean((np.log(np.abs(cab*(1 + 0.2*W*np.sqrt(xb)))) - np.log(deb))**2))
    print('  eta={}: new {:.3f}  vs  old sqrt/W {:.3f}'.format(eta, r_new, r_old))

## Verdict + next step

*(Fill from the printed output.)*

- **Diagnostic:** `ceff2` band pooled-all = [FILL], fixed-eta = [FILL]. `f<=0.3` [is / is not] the obstacle; the split is by **eta**.
- **W-sign:** `W(eta)` = {0.1: 0.412, 0.3: ..., 1.0: 0.072} decreases with eta while measured `c_fs` **increases** -> the published weight has the wrong sign.
- **Two-regime fit:** `x_t = [FILL]`, `p = [FILL]`, `c_fs = {0.1: [FILL], 0.3: [FILL], 1.0: [FILL]}` (monotone in eta: [yes/no]). log-RMS improvement over the old sqrt/W form: [FILL] vs [FILL].
- **Phase 2 (C, rebuild):** implement `ncdm_ceff2_mode = 2` = `two_regime_ceff2(x, ca2, c_fs(eta), x_t, p)` in `perturbations_ceff2_ncdm`, capped at 1/3 for stability, with `x_t`, `p`, and a `c_fs(eta)` parametrization from this fit.
- **Phase 3 (decisive):** fluid mode-2 vs exact, **P(k) residual over k<=1 only**, at `f=0.3, eta=0.1`. Success = ~1%. If it fails even at k<=1 -> exact + q(f) schedule (notebook 14).

**Keep in sync:** `fluid_closure_helpers.py` (unit-tested); `smooth_step`/`two_regime_ceff2` here must match the future C `ncdm_ceff2_mode = 2`.